# 🏆 BTK Datathon 2026 — v5 PRODUCTION (Final)

**Yarışma Metriği:** MSE  
**Donanım:** NVIDIA CUDA GPU  
**Felsefe:** *Trust CV. Build robust. Validate independently.*

## v5'teki Güvenlik Katmanları

| Katman | Amaç |
|--------|------|
| **80/20 Hold-out** | Bağımsız doğrulama (public LB simülatörü) |
| **Feature Selection** | Gürültülü feature'ları temizle |
| **Küçük MLP** | Overfit önle (75k→~10k parametre) |
| **KNN top-30 features** | Curse of dimensionality önle |
| **Ridge top-10 poly** | Polynomial boyut patlamasını sınırla |
| **Optuna 60/60/40 trials** | Validation leak azalt |
| **Pseudo-labeling A/B test** | Sadece hold-out'ta iyileştirirse kullan |
| **Stability check** | Per-fold MSE varyans kontrolü |

**Akış:**
```
1. Train → 80% dev + 20% holdout (stratified)
2. Tüm model geliştirme dev üzerinde (Optuna, CV, stacking)
3. Holdout MSE bir kere ölç → CV ile karşılaştır
4. Pseudo-labeling A/B test: hold-out hangisinde daha iyi?
5. Final için TÜM 10000 ile yeniden eğit, test'e predict et
```

In [ ]:
# ── Paket kurulumu (Colab için) ──────────────────────────────────────────────
!pip install sentence-transformers -q 2>&1 | tail -1
!pip install xgboost lightgbm catboost optuna -q 2>&1 | tail -1

In [ ]:
import pandas as pd
import numpy as np
import warnings, gc, time, os
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
from optuna.pruners import SuccessiveHalvingPruner, MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED       = 42
N_FOLDS    = 10
N_SEEDS    = 3
HOLDOUT_RATIO = 0.2
TARGET_COL = 'career_success_score'

np.random.seed(SEED)
print('✓ Kütüphaneler hazır')

In [ ]:
# ── GPU Kontrol ──────────────────────────────────────────────────────────────
GPU_AVAILABLE = False; LGB_GPU = False; CAT_GPU = False; BERT_DEVICE = 'cpu'
tx = np.random.rand(100,5).astype(np.float32); ty = np.random.rand(100).astype(np.float32)

try:
    m = xgb.XGBRegressor(n_estimators=10, device='cuda', tree_method='hist'); m.fit(tx, ty)
    GPU_AVAILABLE = True; print('✓ XGBoost GPU')
except: print('✗ XGBoost CPU')

try:
    m = lgb.LGBMRegressor(n_estimators=10, device='gpu', verbose=-1); m.fit(tx, ty)
    LGB_GPU = True; print('✓ LightGBM GPU')
except: print('✗ LightGBM CPU')

try:
    m = CatBoostRegressor(iterations=10, task_type='GPU', devices='0', verbose=0); m.fit(tx, ty)
    CAT_GPU = True; print('✓ CatBoost GPU')
except: print('✗ CatBoost CPU')

try:
    import torch
    if torch.cuda.is_available():
        BERT_DEVICE = 'cuda'; print(f'✓ PyTorch CUDA: {torch.cuda.get_device_name(0)}')
    else: print('✗ PyTorch CPU')
except: print('✗ PyTorch yok')

xgb_gpu = lambda: {'device':'cuda','tree_method':'hist'} if GPU_AVAILABLE else {'tree_method':'hist','n_jobs':-1}
lgb_gpu = lambda: {'device':'gpu'} if LGB_GPU else {'n_jobs':-1}
cat_gpu = lambda: {'task_type':'GPU','devices':'0'} if CAT_GPU else {}

## 1. Veri + 🔒 80/20 Hold-out Split

In [ ]:
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test_x.csv'
SAMPLE_PATH = 'sample_submission.csv'

train_full = pd.read_csv(TRAIN_PATH)
test       = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

y_full = train_full[TARGET_COL].values
y_bin_full = pd.qcut(y_full, q=10, labels=False, duplicates='drop')

# 🔒 STRATIFIED 80/20 SPLIT — diagram'daki gibi
dev_idx, holdout_idx = train_test_split(
    np.arange(len(train_full)),
    test_size=HOLDOUT_RATIO,
    random_state=SEED,
    stratify=y_bin_full
)

train_dev  = train_full.iloc[dev_idx].reset_index(drop=True)
train_hold = train_full.iloc[holdout_idx].reset_index(drop=True)
y_dev      = y_full[dev_idx]
y_hold     = y_full[holdout_idx]

print(f'Full train: {train_full.shape}')
print(f'Dev      : {train_dev.shape}  (model burada kurulacak)')
print(f'Holdout  : {train_hold.shape}  (sadece final check için)')
print(f'Test     : {test.shape}')
print(f'\nDev μ={y_dev.mean():.2f} σ={y_dev.std():.2f}')
print(f'Hold μ={y_hold.mean():.2f} σ={y_hold.std():.2f}  ← yakın olmalı')

# CV split'leri dev için
y_dev_bin = pd.qcut(y_dev, q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(train_dev, y_dev_bin))
test_ids = test['student_id'].values

## 2. 🎯 Role-Skill Engineering

In [ ]:
ROLE_PROFILES = {
    'Cloud Engineer':        {'cloud_score':3, 'devops_score':2, 'problem_solving_score':1},
    'DevOps Engineer':       {'devops_score':3, 'cloud_score':2, 'problem_solving_score':1},
    'MLOps Engineer':        {'devops_score':2, 'machine_learning_score':2, 'cloud_score':1, 'problem_solving_score':1},
    'Frontend Developer':    {'frontend_score':3, 'coding_score':2, 'problem_solving_score':1},
    'Backend Developer':     {'backend_score':3, 'coding_score':2, 'sql_score':1, 'problem_solving_score':1},
    'Software Developer':    {'coding_score':2, 'problem_solving_score':2, 'data_structures_score':2, 'backend_score':1, 'frontend_score':1},
    'Data Scientist':        {'machine_learning_score':3, 'sql_score':2, 'problem_solving_score':2, 'data_structures_score':1},
    'AI Engineer':           {'machine_learning_score':3, 'coding_score':2, 'problem_solving_score':1, 'data_structures_score':1},
    'Data Analyst':          {'sql_score':3, 'data_structures_score':1, 'problem_solving_score':1},
    'Product Analyst':       {'sql_score':2, 'problem_solving_score':2},
    'Cybersecurity Analyst': {'devops_score':2, 'problem_solving_score':2, 'coding_score':1},
}
PRIMARY_SKILL = {r: max(w.items(), key=lambda x: x[1])[0] for r, w in ROLE_PROFILES.items()}

def add_role_features(df):
    df = df.copy()
    df['matched_skill'] = 0.0
    for role, skill in PRIMARY_SKILL.items():
        m = df['target_role'] == role
        df.loc[m, 'matched_skill'] = df.loc[m, skill].values
    df['role_composite'] = 0.0
    for role, w in ROLE_PROFILES.items():
        m = df['target_role'] == role
        if m.sum() == 0: continue
        tot = sum(w.values())
        df.loc[m, 'role_composite'] = (sum(df.loc[m, sk]*wt for sk, wt in w.items()) / tot).values
    return df

train_dev  = add_role_features(train_dev)
train_hold = add_role_features(train_hold)
test       = add_role_features(test)
print('✓ Role-skill features eklendi (dev, holdout, test)')

## 3. 🧠 NLP — BERT + TF-IDF (full corpus üzerinde fit)

In [ ]:
BERT_OK = False
all_texts = pd.concat([
    train_dev['mentor_feedback_text'],
    train_hold['mentor_feedback_text'],
    test['mentor_feedback_text']
]).fillna('').reset_index(drop=True)
n_dev, n_hold, n_test = len(train_dev), len(train_hold), len(test)

try:
    from sentence_transformers import SentenceTransformer
    print('Loading Sentence-BERT...')
    bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=BERT_DEVICE)
    print(f'Encoding {len(all_texts)} texts on {BERT_DEVICE}...')
    t0 = time.time()
    emb = bert_model.encode(all_texts.tolist(), batch_size=128, show_progress_bar=True,
                              convert_to_numpy=True, device=BERT_DEVICE)
    print(f'BERT: {emb.shape} in {time.time()-t0:.1f}s')
    pca = PCA(n_components=64, random_state=SEED)
    bert_reduced = pca.fit_transform(emb)
    print(f'PCA → 64-dim, explained: {pca.explained_variance_ratio_.sum():.3f}')
    
    bert_cols = [f'bert_{i}' for i in range(64)]
    train_bert      = pd.DataFrame(bert_reduced[:n_dev], columns=bert_cols)
    train_bert_hold = pd.DataFrame(bert_reduced[n_dev:n_dev+n_hold], columns=bert_cols)
    test_bert       = pd.DataFrame(bert_reduced[n_dev+n_hold:], columns=bert_cols)
    
    del bert_model, emb; gc.collect()
    if BERT_DEVICE == 'cuda':
        import torch; torch.cuda.empty_cache()
    BERT_OK = True
except Exception as e:
    print(f'⚠ BERT skip: {str(e)[:80]}')
    train_bert = pd.DataFrame(); train_bert_hold = pd.DataFrame(); test_bert = pd.DataFrame()

In [ ]:
# TF-IDF word + char
word_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3, sublinear_tf=True, analyzer='word')
word_m = word_tfidf.fit_transform(all_texts)
word_feats = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(word_m)

char_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(3,5), min_df=3, sublinear_tf=True, analyzer='char_wb')
char_m = char_tfidf.fit_transform(all_texts)
char_feats = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(char_m)

nlp_cols = [f'nlp_w_{i}' for i in range(50)] + [f'nlp_c_{i}' for i in range(30)]
nlp_all = pd.DataFrame(np.hstack([word_feats, char_feats]), columns=nlp_cols)
nlp_all['text_len']   = all_texts.str.len().values
nlp_all['word_count'] = all_texts.str.split().str.len().values

train_nlp      = nlp_all.iloc[:n_dev].reset_index(drop=True)
train_nlp_hold = nlp_all.iloc[n_dev:n_dev+n_hold].reset_index(drop=True)
test_nlp       = nlp_all.iloc[n_dev+n_hold:].reset_index(drop=True)
print(f'✓ TF-IDF features: {train_nlp.shape[1]}')

## 4. Target Encoding (sızıntısız, dev y kullanılır)

In [ ]:
def kfold_te(df_dev, df_hold, df_test, col, y_dev, splits, smoothing=20):
    """Dev'de OOF target encoding, holdout+test full dev kullanır."""
    gm = y_dev.mean()
    dev_temp = df_dev.copy()
    dev_temp['__y__'] = y_dev
    
    oof = np.zeros(len(df_dev))
    for tr_idx, val_idx in splits:
        tr = dev_temp.iloc[tr_idx]
        agg = tr.groupby(col)['__y__'].agg(['mean','count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = dev_temp.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    
    agg = dev_temp.groupby(col)['__y__'].agg(['mean','count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    hold_enc = df_hold[col].map(agg['s']).fillna(gm).values
    test_enc = df_test[col].map(agg['s']).fillna(gm).values
    return oof, hold_enc, test_enc

te_train = {}; te_hold = {}; te_test = {}

# Tek değişkenli
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, ho, te = kfold_te(train_dev, train_hold, test, col, y_dev, FOLD_SPLITS, 20)
    te_train[f'te_{col}'] = oo; te_hold[f'te_{col}'] = ho; te_test[f'te_{col}'] = te

# Çapraz
for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_dev[combo]  = train_dev[c1].astype(str)  + '|' + train_dev[c2].astype(str)
    train_hold[combo] = train_hold[c1].astype(str) + '|' + train_hold[c2].astype(str)
    test[combo]       = test[c1].astype(str)       + '|' + test[c2].astype(str)
    oo, ho, te = kfold_te(train_dev, train_hold, test, combo, y_dev, FOLD_SPLITS, 30)
    te_train[f'te_{combo}'] = oo; te_hold[f'te_{combo}'] = ho; te_test[f'te_{combo}'] = te
    train_dev.drop(columns=[combo], inplace=True)
    train_hold.drop(columns=[combo], inplace=True)
    test.drop(columns=[combo], inplace=True)

train_te = pd.DataFrame(te_train).reset_index(drop=True)
hold_te  = pd.DataFrame(te_hold).reset_index(drop=True)
test_te  = pd.DataFrame(te_test).reset_index(drop=True)
print(f'✓ Target encoding features: {train_te.shape[1]}')

## 5. Feature Engineering — Tam Pipeline

In [ ]:
TIER_MAP  = {'Tier 1':4,'Tier 2':3,'Tier 3':2,'Tier 4':1}
TECH_COLS = ['coding_score','problem_solving_score','data_structures_score',
             'sql_score','machine_learning_score','backend_score','frontend_score',
             'cloud_score','devops_score']
SOFT_COLS = ['communication_score','teamwork_score','leadership_score','presentation_score']
CAT_COLS  = ['department','target_role','hobby','preferred_social_media_platform']

def build_features(df_dev, df_hold, df_test):
    df_dev = df_dev.copy(); df_hold = df_hold.copy(); df_test = df_test.copy()
    fill_cols = ['english_exam_score','internship_duration_months','portfolio_score',
                 'github_avg_stars','open_source_contribution_count',
                 'linkedin_profile_score','hr_interview_score']
    medians = {c: df_dev[c].median() for c in fill_cols}  # SADECE DEV'den
    for c in fill_cols:
        df_dev[c]  = df_dev[c].fillna(medians[c])
        df_hold[c] = df_hold[c].fillna(medians[c])
        df_test[c] = df_test[c].fillna(medians[c])
    
    def derive(df):
        df = df.copy()
        df['tech_mean']  = df[TECH_COLS].mean(axis=1)
        df['tech_max']   = df[TECH_COLS].max(axis=1)
        df['tech_min']   = df[TECH_COLS].min(axis=1)
        df['tech_std']   = df[TECH_COLS].std(axis=1)
        df['tech_sum']   = df[TECH_COLS].sum(axis=1)
        df['tech_top3']  = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
        df['tech_bot3']  = df[TECH_COLS].apply(lambda r: r.nsmallest(3).mean(), axis=1)
        df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
        df['soft_sum']  = df[SOFT_COLS].sum(axis=1)
        df['soft_std']  = df[SOFT_COLS].std(axis=1)
        # Role interactions
        df['matched_x_proj']        = df['matched_skill'] * df['project_quality_score']
        df['matched_x_interview']   = df['matched_skill'] * df['technical_interview_score']
        df['role_comp_x_proj']      = df['role_composite'] * df['project_quality_score']
        df['role_comp_x_interview'] = df['role_composite'] * df['technical_interview_score']
        df['matched_minus_tech']    = df['matched_skill'] - df['tech_mean']
        df['role_comp_minus_avg']   = df['role_composite'] - df['tech_mean']
        # Mülakat
        df['interview_composite'] = df['technical_interview_score']*0.6 + df['hr_interview_score']*0.4
        df['interview_diff']      = df['technical_interview_score'] - df['hr_interview_score']
        # Deneyim
        df['total_projects']  = df['real_client_project_count'] + df['freelance_project_count']
        df['experience_score']= (df['internship_count']*3 + df['real_client_project_count']*2 + df['freelance_project_count'])
        df['exp_per_year']    = df['experience_score'] / (df['age'] - 18).clip(lower=1)
        # GitHub
        df['github_score']    = (df['github_repo_count'] * np.log1p(df['github_avg_stars']+1) + df['open_source_contribution_count'])
        df['oss_ratio']       = df['open_source_contribution_count'] / (df['github_repo_count']+1)
        # Profil
        df['profile_composite']= (df['portfolio_score']+df['linkedin_profile_score']+df['cv_quality_score'])/3
        df['profile_min']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].min(axis=1)
        df['profile_max']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].max(axis=1)
        # Hackathon
        df['hackathon_eff']   = np.where(df['hackathon_count']>0, df['hackathon_awards']/df['hackathon_count'], 0)
        df['app_success_rate']= df['interviews_attended'] / (df['applications_sent']+1)
        # Üni
        df['university_tier_num'] = df['university_tier'].map(TIER_MAP)
        df['years_to_grad'] = df['graduation_year'] - df['application_year']
        df['cert_per_internship'] = df['certification_count'] / (df['internship_count']+1)
        df['cgpa_x_attendance']   = df['cgpa'] * df['attendance_rate']
        df['failed_penalty']      = df['failed_courses_count'] / (df['cgpa']+0.01)
        # Genel interactions
        df['soft_x_technical']    = df['soft_mean'] * df['technical_interview_score']
        df['proj_x_interview']    = df['project_quality_score'] * df['technical_interview_score']
        df['proj_x_portfolio']    = df['project_quality_score'] * df['portfolio_score']
        # Composite
        df['overall_score'] = (df['tech_mean']*0.25 + df['soft_mean']*0.15 +
                                df['project_quality_score']*0.25 + df['profile_composite']*0.10 +
                                df['interview_composite']*0.15 + df['matched_skill']*0.10)
        return df
    
    df_dev = derive(df_dev); df_hold = derive(df_hold); df_test = derive(df_test)
    df_dev_cat = df_dev.copy(); df_hold_cat = df_hold.copy(); df_test_cat = df_test.copy()
    
    # Label encoding (combined fit)
    for col in CAT_COLS + ['university_tier']:
        le = LabelEncoder()
        le.fit(pd.concat([df_dev[col], df_hold[col], df_test[col]]).astype(str))
        df_dev[col]  = le.transform(df_dev[col].astype(str))
        df_hold[col] = le.transform(df_hold[col].astype(str))
        df_test[col] = le.transform(df_test[col].astype(str))
    
    drop_cols = ['student_id','mentor_feedback_text']
    if TARGET_COL in df_dev.columns: drop_cols.append(TARGET_COL)
    df_dev  = df_dev.drop(columns=drop_cols, errors='ignore')
    df_hold = df_hold.drop(columns=drop_cols, errors='ignore')
    df_test = df_test.drop(columns=drop_cols, errors='ignore')
    df_dev_cat  = df_dev_cat.drop(columns=drop_cols, errors='ignore')
    df_hold_cat = df_hold_cat.drop(columns=drop_cols, errors='ignore')
    df_test_cat = df_test_cat.drop(columns=drop_cols, errors='ignore')
    
    return df_dev, df_hold, df_test, df_dev_cat, df_hold_cat, df_test_cat

X_dev, X_hold, X_test, X_dev_cat, X_hold_cat, X_test_cat = build_features(train_dev, train_hold, test)

# Birleştir
def merge_parts(base, nlp, te, bert):
    parts = [base.reset_index(drop=True), nlp.reset_index(drop=True), te.reset_index(drop=True)]
    if BERT_OK: parts.append(bert.reset_index(drop=True))
    return pd.concat(parts, axis=1)

X_dev      = merge_parts(X_dev, train_nlp, train_te, train_bert)
X_hold     = merge_parts(X_hold, train_nlp_hold, hold_te, train_bert_hold)
X_test     = merge_parts(X_test, test_nlp, test_te, test_bert)
X_dev_cat  = merge_parts(X_dev_cat, train_nlp, train_te, train_bert)
X_hold_cat = merge_parts(X_hold_cat, train_nlp_hold, hold_te, train_bert_hold)
X_test_cat = merge_parts(X_test_cat, test_nlp, test_te, test_bert)

# Null doldurma (dev medianı)
col_medians = X_dev.median(numeric_only=True)
X_dev  = X_dev.fillna(col_medians)
X_hold = X_hold.fillna(col_medians)
X_test = X_test.fillna(col_medians)
for c in X_dev_cat.columns:
    if pd.api.types.is_numeric_dtype(X_dev_cat[c]) and X_dev_cat[c].isna().any():
        X_dev_cat[c]  = X_dev_cat[c].fillna(col_medians.get(c, 0))
        X_hold_cat[c] = X_hold_cat[c].fillna(col_medians.get(c, 0))
        X_test_cat[c] = X_test_cat[c].fillna(col_medians.get(c, 0))

cat_features_idx = [X_dev_cat.columns.get_loc(c) for c in CAT_COLS + ['university_tier']]
print(f'✓ Dev: {X_dev.shape}, Holdout: {X_hold.shape}, Test: {X_test.shape}')

## 6. 🔍 Feature Selection (XGB Importance)

In [ ]:
# Quick XGB to get importances
print('Quick XGB ile feature importance hesaplanıyor...')
t0 = time.time()
quick = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=SEED, **xgb_gpu())
quick.fit(X_dev, y_dev)
imps = pd.Series(quick.feature_importances_, index=X_dev.columns).sort_values(ascending=False)
print(f'  Done in {time.time()-t0:.1f}s')

# Tüm modeller için: top %85 feature
all_features_keep = imps[imps > imps.quantile(0.15)].index.tolist()
print(f'  Tüm modeller için: {X_dev.shape[1]} → {len(all_features_keep)} feature')

# KNN için: top 30 feature
top_30_features = imps.head(30).index.tolist()
print(f'  KNN için: top 30 feature')

# Ridge+Poly için: top 10 feature
top_10_features = imps.head(10).index.tolist()
print(f'  Ridge+Poly için: top 10 feature')
print(f'  Top 10: {top_10_features}')

# Standardize (MLP, KNN için)
scaler = StandardScaler()
X_dev_std  = pd.DataFrame(scaler.fit_transform(X_dev), columns=X_dev.columns, index=X_dev.index)
X_hold_std = pd.DataFrame(scaler.transform(X_hold), columns=X_hold.columns, index=X_hold.index)
X_test_std = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 7. Optuna — Reduced Trials (Validation Leak Azalt)

In [ ]:
# Hızlı 3-fold (Optuna için)
opt_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
OPT_SPLITS = list(opt_skf.split(X_dev, y_dev_bin))

# Selected features for tree models (top 85%)
X_dev_sel      = X_dev[all_features_keep]
X_hold_sel     = X_hold[all_features_keep]
X_test_sel     = X_test[all_features_keep]
X_dev_cat_sel  = X_dev_cat[all_features_keep]
X_hold_cat_sel = X_hold_cat[all_features_keep]
X_test_cat_sel = X_test_cat[all_features_keep]
cat_features_idx_sel = [X_dev_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_dev_cat_sel.columns]
print(f'Tree modeller için aktif feature: {len(all_features_keep)}')

In [ ]:
# XGBoost
def xgb_obj(trial):
    params = {'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'early_stopping_rounds': 50, 'random_state': SEED, **xgb_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = xgb.XGBRegressor(**params)
        m.fit(X_dev_sel.iloc[tr], y_dev[tr], eval_set=[(X_dev_sel.iloc[va], y_dev[va])], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('XGBoost (60 trial)...')
t0 = time.time()
xgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                  pruner=SuccessiveHalvingPruner())
xgb_study.optimize(xgb_obj, n_trials=60, show_progress_bar=True)
print(f'XGB: {xgb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# LightGBM
def lgb_obj(trial):
    params = {'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'random_state': SEED, 'verbose': -1, **lgb_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_dev_sel.iloc[tr], y_dev[tr], eval_set=[(X_dev_sel.iloc[va], y_dev[va])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('LightGBM (60 trial)...')
t0 = time.time()
lgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=SuccessiveHalvingPruner())
lgb_study.optimize(lgb_obj, n_trials=60, show_progress_bar=True)
print(f'LGB: {lgb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# CatBoost
def cat_obj(trial):
    params = {'iterations': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength': trial.suggest_float('random_strength', 0.5, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': SEED, 'verbose': 0, 'loss_function': 'RMSE',
        'early_stopping_rounds': 50, **cat_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = CatBoostRegressor(**params)
        m.fit(X_dev_cat_sel.iloc[tr], y_dev[tr], eval_set=(X_dev_cat_sel.iloc[va], y_dev[va]),
              cat_features=cat_features_idx_sel, verbose=0)
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_cat_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('CatBoost (40 trial)...')
t0 = time.time()
cat_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=SuccessiveHalvingPruner())
cat_study.optimize(cat_obj, n_trials=40, show_progress_bar=True)
print(f'CAT: {cat_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# HistGradientBoosting
def hgb_obj(trial):
    params = {'max_iter': trial.suggest_int('max_iter', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 300),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10, log=True),
        'random_state': SEED}
    rmses = []
    for tr, va in OPT_SPLITS:
        m = HistGradientBoostingRegressor(**params); m.fit(X_dev_sel.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
    return np.mean(rmses)

print('HistGB (25 trial)...')
t0 = time.time()
hgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=MedianPruner(n_startup_trials=5))
hgb_study.optimize(hgb_obj, n_trials=25, show_progress_bar=True)
print(f'HGB: {hgb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# MLP — KÜÇÜK mimari (overfit önle)
MLP_ARCHS = [(64,), (128,), (64, 32), (128, 64), (96, 48)]

def mlp_obj(trial):
    arch_idx = trial.suggest_int('arch_idx', 0, len(MLP_ARCHS)-1)
    params = {'hidden_layer_sizes': MLP_ARCHS[arch_idx],
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('lr_init', 1e-4, 5e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),
        'activation': 'relu', 'solver': 'adam', 'max_iter': 200,
        'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15,
        'random_state': SEED}
    rmses = []
    for tr, va in OPT_SPLITS:
        m = MLPRegressor(**params); m.fit(X_dev_std.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_std.iloc[va]))))
    return np.mean(rmses)

print('MLP small (25 trial)...')
t0 = time.time()
mlp_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=MedianPruner(n_startup_trials=5))
mlp_study.optimize(mlp_obj, n_trials=25, show_progress_bar=True)
print(f'MLP: {mlp_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# Ridge + Polynomial (top 10 feature)
X_dev_poly  = X_dev[top_10_features].copy()
X_hold_poly = X_hold[top_10_features].copy()
X_test_poly = X_test[top_10_features].copy()

def ridge_obj(trial):
    params = {'alpha': trial.suggest_float('alpha', 0.1, 100, log=True),
              'degree': trial.suggest_int('degree', 2, 3)}
    rmses = []
    for tr, va in OPT_SPLITS:
        pipe = Pipeline([('scaler', StandardScaler()),
                          ('poly', PolynomialFeatures(degree=params['degree'], include_bias=False)),
                          ('ridge', Ridge(alpha=params['alpha'], random_state=SEED))])
        pipe.fit(X_dev_poly.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], pipe.predict(X_dev_poly.iloc[va]))))
    return np.mean(rmses)

print('Ridge+Poly (15 trial)...')
t0 = time.time()
ridge_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
ridge_study.optimize(ridge_obj, n_trials=15, show_progress_bar=True)
print(f'Ridge+Poly: {ridge_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# KNN — sadece top 30 feature (curse of dimensionality önle)
X_dev_knn  = X_dev_std[top_30_features]
X_hold_knn = X_hold_std[top_30_features]
X_test_knn = X_test_std[top_30_features]

def knn_obj(trial):
    params = {'n_neighbors': trial.suggest_int('n_neighbors', 10, 100),
              'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
              'p': trial.suggest_int('p', 1, 2)}
    rmses = []
    for tr, va in OPT_SPLITS:
        m = KNeighborsRegressor(**params, n_jobs=-1); m.fit(X_dev_knn.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_knn.iloc[va]))))
    return np.mean(rmses)

print('KNN top-30 (15 trial)...')
t0 = time.time()
knn_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
knn_study.optimize(knn_obj, n_trials=15, show_progress_bar=True)
print(f'KNN: {knn_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

## 8. Stacking — Dev Üzerinde 10-Fold + Holdout/Test Tahminleri

In [ ]:
def stack_with_holdout(factory, splits, X_tr, X_ho, X_te, y_tr,
                         n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    """OOF + holdout + test predictions üret"""
    oof = np.zeros(len(y_tr))
    holdout_p = np.zeros(len(X_ho))
    test_p = np.zeros(len(X_te))
    per_fold_mse = []
    
    for seed in range(n_seeds):
        fold_ho = np.zeros(len(X_ho))
        fold_te = np.zeros(len(X_te))
        for fold_i, (tr_idx, val_idx) in enumerate(splits):
            m = factory(seed)
            if kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx],
                      eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx],
                      eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx],
                      eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
            else:
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
            val_pred = m.predict(X_tr.iloc[val_idx])
            oof[val_idx] += val_pred / n_seeds
            fold_ho += m.predict(X_ho) / len(splits)
            fold_te += m.predict(X_te) / len(splits)
            if seed == 0:
                per_fold_mse.append(mean_squared_error(y_tr[val_idx], val_pred))
        holdout_p += fold_ho / n_seeds
        test_p += fold_te / n_seeds
    return oof, holdout_p, test_p, per_fold_mse

print(f'Stacking ({N_FOLDS}-fold × {N_SEEDS}-seed × 7 model)...\n')
stability = {}

t0 = time.time()
xgb_factory = lambda s: xgb.XGBRegressor(**xgb_study.best_params, random_state=SEED+s,
    **xgb_gpu(), early_stopping_rounds=50, n_estimators=3000)
xgb_oof, xgb_ho, xgb_te, xgb_fmse = stack_with_holdout(xgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel, y_dev, kind='xgb')
stability['XGB'] = xgb_fmse
print(f'  XGB : OOF MSE={mean_squared_error(y_dev, xgb_oof):.3f}, Hold MSE={mean_squared_error(y_hold, xgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lgb_factory = lambda s: lgb.LGBMRegressor(**lgb_study.best_params, random_state=SEED+s,
    **lgb_gpu(), n_estimators=3000, verbose=-1)
lgb_oof, lgb_ho, lgb_te, lgb_fmse = stack_with_holdout(lgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel, y_dev, kind='lgb')
stability['LGB'] = lgb_fmse
print(f'  LGB : OOF MSE={mean_squared_error(y_dev, lgb_oof):.3f}, Hold MSE={mean_squared_error(y_hold, lgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
cat_factory = lambda s: CatBoostRegressor(**cat_study.best_params, random_seed=SEED+s,
    **cat_gpu(), iterations=3000, verbose=0, loss_function='RMSE', early_stopping_rounds=50)
cat_oof, cat_ho, cat_te, cat_fmse = stack_with_holdout(cat_factory, FOLD_SPLITS, X_dev_cat_sel, X_hold_cat_sel, X_test_cat_sel, y_dev, kind='cat', cat_idx=cat_features_idx_sel)
stability['CAT'] = cat_fmse
print(f'  CAT : OOF MSE={mean_squared_error(y_dev, cat_oof):.3f}, Hold MSE={mean_squared_error(y_hold, cat_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
hgb_factory = lambda s: HistGradientBoostingRegressor(**hgb_study.best_params, random_state=SEED+s)
hgb_oof, hgb_ho, hgb_te, hgb_fmse = stack_with_holdout(hgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel, y_dev)
stability['HGB'] = hgb_fmse
print(f'  HGB : OOF MSE={mean_squared_error(y_dev, hgb_oof):.3f}, Hold MSE={mean_squared_error(y_hold, hgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
best_mlp = mlp_study.best_params
mlp_factory = lambda s: MLPRegressor(
    hidden_layer_sizes=MLP_ARCHS[best_mlp['arch_idx']],
    alpha=best_mlp['alpha'], learning_rate_init=best_mlp['lr_init'],
    batch_size=best_mlp['batch_size'], activation='relu', solver='adam',
    max_iter=300, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, random_state=SEED+s)
mlp_oof, mlp_ho, mlp_te, mlp_fmse = stack_with_holdout(mlp_factory, FOLD_SPLITS, X_dev_std, X_hold_std, X_test_std, y_dev)
stability['MLP'] = mlp_fmse
print(f'  MLP : OOF MSE={mean_squared_error(y_dev, mlp_oof):.3f}, Hold MSE={mean_squared_error(y_hold, mlp_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
best_ridge = ridge_study.best_params
ridge_factory = lambda s: Pipeline([('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=best_ridge['degree'], include_bias=False)),
    ('ridge', Ridge(alpha=best_ridge['alpha'], random_state=SEED+s))])
ridge_oof, ridge_ho, ridge_te, ridge_fmse = stack_with_holdout(ridge_factory, FOLD_SPLITS, X_dev_poly, X_hold_poly, X_test_poly, y_dev, n_seeds=1)
stability['Ridge'] = ridge_fmse
print(f'  Ridge+Poly : OOF MSE={mean_squared_error(y_dev, ridge_oof):.3f}, Hold MSE={mean_squared_error(y_hold, ridge_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
knn_factory = lambda s: KNeighborsRegressor(**knn_study.best_params, n_jobs=-1)
knn_oof, knn_ho, knn_te, knn_fmse = stack_with_holdout(knn_factory, FOLD_SPLITS, X_dev_knn, X_hold_knn, X_test_knn, y_dev, n_seeds=1)
stability['KNN'] = knn_fmse
print(f'  KNN top-30 : OOF MSE={mean_squared_error(y_dev, knn_oof):.3f}, Hold MSE={mean_squared_error(y_hold, knn_ho):.3f}  ({time.time()-t0:.0f}s)')

oof_matrix      = np.column_stack([xgb_oof, lgb_oof, cat_oof, hgb_oof, mlp_oof, ridge_oof, knn_oof])
holdout_matrix  = np.column_stack([xgb_ho, lgb_ho, cat_ho, hgb_ho, mlp_ho, ridge_ho, knn_ho])
test_matrix     = np.column_stack([xgb_te, lgb_te, cat_te, hgb_te, mlp_te, ridge_te, knn_te])
model_names     = ['XGB','LGB','CAT','HGB','MLP','Ridge','KNN']

## 9. 🔬 Stability Check

In [ ]:
print('Per-fold MSE varyans analizi (yüksek = unstable model):\n')
for name, fmse in stability.items():
    mean_mse = np.mean(fmse)
    std_mse = np.std(fmse)
    cv = std_mse / mean_mse * 100
    flag = '✓ STABLE' if cv < 5 else '⚠️ UNSTABLE' if cv < 10 else '🚨 VERY UNSTABLE'
    print(f'  {name:<8} | mean={mean_mse:6.3f} | std={std_mse:5.3f} | CV={cv:5.2f}%  {flag}')

## 10. Weight Optimization + Holdout Doğrulama

In [ ]:
def weighted_rmse(w, preds, y_true):
    return np.sqrt(mean_squared_error(y_true, preds @ w))

n_models = oof_matrix.shape[1]
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.0, 1.0)] * n_models

# SLSQP weighted
best_loss, best_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_loss:
        best_loss = res.fun; best_w = res.x

# Ridge meta
best_ridge_alpha = None; best_ridge_loss = float('inf')
meta_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for alpha in [0.01, 0.1, 1, 5, 10, 50, 100]:
    fold_mses = []
    for tr_idx, val_idx in meta_kf.split(oof_matrix, y_dev_bin):
        meta = Ridge(alpha=alpha); meta.fit(oof_matrix[tr_idx], y_dev[tr_idx])
        fold_mses.append(np.sqrt(mean_squared_error(y_dev[val_idx], meta.predict(oof_matrix[val_idx]))))
    avg = np.mean(fold_mses)
    if avg < best_ridge_loss: best_ridge_loss = avg; best_ridge_alpha = alpha

ridge_meta = Ridge(alpha=best_ridge_alpha); ridge_meta.fit(oof_matrix, y_dev)

# Compare on holdout
slsqp_ho_mse  = mean_squared_error(y_hold, holdout_matrix @ best_w)
ridge_ho_mse  = mean_squared_error(y_hold, ridge_meta.predict(holdout_matrix))
blend_ho_mse  = mean_squared_error(y_hold, 0.5*(holdout_matrix @ best_w) + 0.5*ridge_meta.predict(holdout_matrix))

print('═'*60)
print('ENSEMBLE YÖNTEMLERİ — Dev OOF vs Holdout MSE')
print('═'*60)
print(f'{"Yöntem":<22} {"Dev OOF":>10} {"Holdout":>10}')
print('-'*60)
print(f'{"SLSQP weighted":<22} {best_loss**2:>10.3f} {slsqp_ho_mse:>10.3f}')
print(f'{"Ridge meta (α="+str(best_ridge_alpha)+")":<22} {best_ridge_loss**2:>10.3f} {ridge_ho_mse:>10.3f}')
print(f'{"50/50 Blend":<22} {"-":>10} {blend_ho_mse:>10.3f}')
print('═'*60)

# En iyi yöntem (HOLDOUT'a göre seç — diagram felsefesi)
options = [
    ('SLSQP', slsqp_ho_mse, lambda M: M @ best_w),
    ('Ridge', ridge_ho_mse, lambda M: ridge_meta.predict(M)),
    ('Blend', blend_ho_mse, lambda M: 0.5*(M @ best_w) + 0.5*ridge_meta.predict(M))
]
options.sort(key=lambda x: x[1])
best_method_name, best_method_mse, best_method_fn = options[0]
print(f'\n✓ En iyi yöntem: {best_method_name} (Holdout MSE = {best_method_mse:.3f})')

## 11. 🔄 Pseudo-Labeling A/B Test (Hold-out üzerinde validate)

In [ ]:
# Konservatif pseudo: sadece %10 en güvenilir test örneği
test_stds = test_matrix.std(axis=1)
PSEUDO_FRAC = 0.10
n_pseudo = int(len(test) * PSEUDO_FRAC)
conf_idx = np.argsort(test_stds)[:n_pseudo]

# Mevcut ensemble tahmini (best_method_fn ile)
ensemble_test_pred = best_method_fn(test_matrix)
pseudo_labels = np.clip(ensemble_test_pred[conf_idx], 0, 100)

print(f'Pseudo-label: {n_pseudo} test örneği (sadece en güvenilir %{PSEUDO_FRAC*100:.0f})')

# Augmented dev
X_dev_aug      = pd.concat([X_dev_sel, X_test_sel.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_cat_aug  = pd.concat([X_dev_cat_sel, X_test_cat_sel.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_std_aug  = pd.concat([X_dev_std, X_test_std.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_poly_aug = pd.concat([X_dev_poly, X_test_poly.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_knn_aug  = pd.concat([X_dev_knn, X_test_knn.iloc[conf_idx]], axis=0).reset_index(drop=True)
y_dev_aug      = np.concatenate([y_dev, pseudo_labels])
y_dev_aug_bin  = pd.qcut(y_dev_aug, q=10, labels=False, duplicates='drop')
FOLD_SPLITS_AUG = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_dev_aug, y_dev_aug_bin))

print(f'Augmented dev: {len(X_dev_aug)} satır (orig {len(X_dev_sel)} + {n_pseudo} pseudo)\n')

In [ ]:
# Augmented stacking
print('Pseudo-augmented stacking...\n')
n_orig = len(y_dev)

t0 = time.time()
xgb_oof2, xgb_ho2, xgb_te2, _ = stack_with_holdout(xgb_factory, FOLD_SPLITS_AUG, X_dev_aug, X_hold_sel, X_test_sel, y_dev_aug, kind='xgb')
lgb_oof2, lgb_ho2, lgb_te2, _ = stack_with_holdout(lgb_factory, FOLD_SPLITS_AUG, X_dev_aug, X_hold_sel, X_test_sel, y_dev_aug, kind='lgb')
cat_oof2, cat_ho2, cat_te2, _ = stack_with_holdout(cat_factory, FOLD_SPLITS_AUG, X_dev_cat_aug, X_hold_cat_sel, X_test_cat_sel, y_dev_aug, kind='cat', cat_idx=cat_features_idx_sel)
hgb_oof2, hgb_ho2, hgb_te2, _ = stack_with_holdout(hgb_factory, FOLD_SPLITS_AUG, X_dev_aug, X_hold_sel, X_test_sel, y_dev_aug)
mlp_oof2, mlp_ho2, mlp_te2, _ = stack_with_holdout(mlp_factory, FOLD_SPLITS_AUG, X_dev_std_aug, X_hold_std, X_test_std, y_dev_aug)
ridge_oof2, ridge_ho2, ridge_te2, _ = stack_with_holdout(ridge_factory, FOLD_SPLITS_AUG, X_dev_poly_aug, X_hold_poly, X_test_poly, y_dev_aug, n_seeds=1)
knn_oof2, knn_ho2, knn_te2, _ = stack_with_holdout(knn_factory, FOLD_SPLITS_AUG, X_dev_knn_aug, X_hold_knn, X_test_knn, y_dev_aug, n_seeds=1)
print(f'Aug stacking: {time.time()-t0:.0f}s')

# Sadece orig OOF kısmı
oof_matrix_aug = np.column_stack([xgb_oof2[:n_orig], lgb_oof2[:n_orig], cat_oof2[:n_orig],
                                   hgb_oof2[:n_orig], mlp_oof2[:n_orig], ridge_oof2[:n_orig], knn_oof2[:n_orig]])
holdout_matrix_aug = np.column_stack([xgb_ho2, lgb_ho2, cat_ho2, hgb_ho2, mlp_ho2, ridge_ho2, knn_ho2])
test_matrix_aug    = np.column_stack([xgb_te2, lgb_te2, cat_te2, hgb_te2, mlp_te2, ridge_te2, knn_te2])

# Aug için weight opt
best_aug_loss, best_aug_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix_aug, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_aug_loss:
        best_aug_loss = res.fun; best_aug_w = res.x

ridge_meta_aug = Ridge(alpha=best_ridge_alpha); ridge_meta_aug.fit(oof_matrix_aug, y_dev)

# Holdout MSE'leri
aug_slsqp_ho = mean_squared_error(y_hold, holdout_matrix_aug @ best_aug_w)
aug_ridge_ho = mean_squared_error(y_hold, ridge_meta_aug.predict(holdout_matrix_aug))
aug_blend_ho = mean_squared_error(y_hold, 0.5*(holdout_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(holdout_matrix_aug))

print('\n═'*30)
print('  PSEUDO-LABELING A/B TEST (Holdout MSE)')
print('═'*60)
print(f'  ORIJINAL:')
print(f'    SLSQP        : {slsqp_ho_mse:.3f}')
print(f'    Ridge meta   : {ridge_ho_mse:.3f}')
print(f'    Blend        : {blend_ho_mse:.3f}')
print(f'  PSEUDO-AUGMENTED:')
print(f'    SLSQP        : {aug_slsqp_ho:.3f}')
print(f'    Ridge meta   : {aug_ridge_ho:.3f}')
print(f'    Blend        : {aug_blend_ho:.3f}')
print('═'*60)

In [ ]:
# Tüm seçenekleri Holdout MSE'sine göre sırala — DİAGRAM FELSEFESİ
all_options = [
    ('Orig + SLSQP',  slsqp_ho_mse,  lambda: best_method_fn(test_matrix)),
    ('Orig + Ridge',  ridge_ho_mse,  lambda: ridge_meta.predict(test_matrix)),
    ('Orig + Blend',  blend_ho_mse,  lambda: 0.5*(test_matrix @ best_w) + 0.5*ridge_meta.predict(test_matrix)),
    ('Pseudo + SLSQP', aug_slsqp_ho, lambda: test_matrix_aug @ best_aug_w),
    ('Pseudo + Ridge', aug_ridge_ho, lambda: ridge_meta_aug.predict(test_matrix_aug)),
    ('Pseudo + Blend', aug_blend_ho, lambda: 0.5*(test_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(test_matrix_aug)),
]
all_options.sort(key=lambda x: x[1])

print('\n--- TÜM SEÇENEKLER (Holdout MSE'ye göre sıralı) ---')
for name, mse, _ in all_options:
    print(f'  {name:<20}: {mse:.3f}')

best_name, best_holdout_mse, best_test_pred_fn = all_options[0]
print(f'\n🏆 SEÇİLEN: {best_name}  (Holdout MSE = {best_holdout_mse:.3f})')

final_test_phase1 = np.clip(best_test_pred_fn(), 0, 100)

## 12. 🚀 Phase 4 — Final Retrain on FULL 10000

Dev'de bulunan hiperparametrelerle, TÜM 10000 train üzerinde modelleri yeniden eğit. Hold-out artık 'görülmüş' sayılır → final için daha fazla veri = daha güçlü model.

In [ ]:
# Tüm 10000 train için yeniden feature engineering
print('Full data ile feature engineering...')
train_full_role = add_role_features(train_full)

# Full text corpus (zaten fit'ledik, sadece transform)
# Aslında BERT ve TF-IDF zaten tüm corpus üzerinde fit'lendi (dev+holdout+test)
# Sadece dev+holdout = full train kısmını alıp birleştir
n_dev_orig = n_dev

# Tüm train için NLP feature'larını eski indekslerden topla
if BERT_OK:
    train_full_bert = pd.concat([train_bert, train_bert_hold], axis=0).reset_index(drop=True)
    # Ama orijinal indeks sırası dev_idx+holdout_idx, train_full sırası farklı
    # Doğru sıralamaya getir
    full_bert_ordered = pd.DataFrame(index=range(len(train_full)), columns=train_full_bert.columns)
    full_bert_ordered.iloc[dev_idx] = train_bert.values
    full_bert_ordered.iloc[holdout_idx] = train_bert_hold.values
    full_bert_ordered = full_bert_ordered.astype(float)
else:
    full_bert_ordered = pd.DataFrame()

full_nlp_ordered = pd.DataFrame(index=range(len(train_full)), columns=train_nlp.columns)
full_nlp_ordered.iloc[dev_idx] = train_nlp.values
full_nlp_ordered.iloc[holdout_idx] = train_nlp_hold.values
full_nlp_ordered = full_nlp_ordered.astype(float)

# Target encoding — bu defa TAM train üzerinde k-fold
skf_full = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS_FULL = list(skf_full.split(train_full_role, y_bin_full))

te_full = {}; te_test_full = {}
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, _, te = kfold_te(train_full_role, train_full_role.iloc[:1], test, col, y_full, FOLD_SPLITS_FULL, 20)
    te_full[f'te_{col}'] = oo; te_test_full[f'te_{col}'] = te

for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_full_role[combo] = train_full_role[c1].astype(str)+'|'+train_full_role[c2].astype(str)
    test_temp = test.copy(); test_temp[combo] = test_temp[c1].astype(str)+'|'+test_temp[c2].astype(str)
    oo, _, te = kfold_te(train_full_role, train_full_role.iloc[:1], test_temp, combo, y_full, FOLD_SPLITS_FULL, 30)
    te_full[f'te_{combo}'] = oo; te_test_full[f'te_{combo}'] = te
    train_full_role.drop(columns=[combo], inplace=True)

te_full_df = pd.DataFrame(te_full)
te_test_full_df = pd.DataFrame(te_test_full)

# Final features için: build_features kullan (dev parametresi olarak full kullan)
X_full, _, X_test_v2, X_full_cat, _, X_test_cat_v2 = build_features(train_full_role, train_full_role.iloc[:1], test)

X_full      = pd.concat([X_full.reset_index(drop=True), full_nlp_ordered, te_full_df, full_bert_ordered if BERT_OK else pd.DataFrame()], axis=1)
X_test_v2   = pd.concat([X_test_v2.reset_index(drop=True), test_nlp, te_test_full_df, test_bert if BERT_OK else pd.DataFrame()], axis=1)
X_full_cat  = pd.concat([X_full_cat.reset_index(drop=True), full_nlp_ordered, te_full_df, full_bert_ordered if BERT_OK else pd.DataFrame()], axis=1)
X_test_cat_v2 = pd.concat([X_test_cat_v2.reset_index(drop=True), test_nlp, te_test_full_df, test_bert if BERT_OK else pd.DataFrame()], axis=1)

# Fill
X_full = X_full.fillna(X_full.median(numeric_only=True))
X_test_v2 = X_test_v2.fillna(X_full.median(numeric_only=True))
for c in X_full_cat.columns:
    if pd.api.types.is_numeric_dtype(X_full_cat[c]) and X_full_cat[c].isna().any():
        X_full_cat[c] = X_full_cat[c].fillna(X_full.median(numeric_only=True).get(c, 0))
        X_test_cat_v2[c] = X_test_cat_v2[c].fillna(X_full.median(numeric_only=True).get(c, 0))

# Aynı feature'ları seç
use_cols = [c for c in all_features_keep if c in X_full.columns]
X_full_sel = X_full[use_cols]
X_test_v2_sel = X_test_v2[use_cols]
X_full_cat_sel = X_full_cat[use_cols]
X_test_cat_v2_sel = X_test_cat_v2[use_cols]

scaler_full = StandardScaler()
X_full_std = pd.DataFrame(scaler_full.fit_transform(X_full), columns=X_full.columns)
X_test_v2_std = pd.DataFrame(scaler_full.transform(X_test_v2), columns=X_test_v2.columns)

use_knn = [c for c in top_30_features if c in X_full_std.columns]
use_poly = [c for c in top_10_features if c in X_full.columns]
X_full_knn = X_full_std[use_knn]
X_test_v2_knn = X_test_v2_std[use_knn]
X_full_poly = X_full[use_poly]
X_test_v2_poly = X_test_v2[use_poly]

print(f'Full features ready: {X_full_sel.shape}')

In [ ]:
# Final modelleri TAM 10000 ile 10-fold eğit, test'e predict et
print('Final stacking on FULL 10000 train...\n')

def make_oof_full(factory, splits, X_tr, X_te, y_tr, n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    test_p = np.zeros(len(X_te))
    oof = np.zeros(len(y_tr))
    for seed in range(n_seeds):
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = factory(seed)
            if kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
            else:
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

cat_idx_full = [X_full_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_full_cat_sel.columns]

t0 = time.time()
f_xgb_oof, f_xgb_te = make_oof_full(xgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_v2_sel, y_full, kind='xgb')
f_lgb_oof, f_lgb_te = make_oof_full(lgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_v2_sel, y_full, kind='lgb')
f_cat_oof, f_cat_te = make_oof_full(cat_factory, FOLD_SPLITS_FULL, X_full_cat_sel, X_test_cat_v2_sel, y_full, kind='cat', cat_idx=cat_idx_full)
f_hgb_oof, f_hgb_te = make_oof_full(hgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_v2_sel, y_full)
f_mlp_oof, f_mlp_te = make_oof_full(mlp_factory, FOLD_SPLITS_FULL, X_full_std, X_test_v2_std, y_full)
f_ridge_oof, f_ridge_te = make_oof_full(ridge_factory, FOLD_SPLITS_FULL, X_full_poly, X_test_v2_poly, y_full, n_seeds=1)
f_knn_oof, f_knn_te = make_oof_full(knn_factory, FOLD_SPLITS_FULL, X_full_knn, X_test_v2_knn, y_full, n_seeds=1)
print(f'Phase 4 stacking: {time.time()-t0:.0f}s')

oof_full_matrix = np.column_stack([f_xgb_oof, f_lgb_oof, f_cat_oof, f_hgb_oof, f_mlp_oof, f_ridge_oof, f_knn_oof])
test_full_matrix = np.column_stack([f_xgb_te, f_lgb_te, f_cat_te, f_hgb_te, f_mlp_te, f_ridge_te, f_knn_te])

# Aynı meta-learner (best_w veya ridge_meta) Phase 1'den geliyor
# Ama daha güvenli için tekrar fit edebiliriz full OOF üzerinde
best_full_loss, best_full_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_full_matrix, y_full),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_full_loss:
        best_full_loss = res.fun; best_full_w = res.x

ridge_meta_full = Ridge(alpha=best_ridge_alpha)
ridge_meta_full.fit(oof_full_matrix, y_full)

final_full_oof = oof_full_matrix @ best_full_w
final_full_oof_mse = mean_squared_error(y_full, final_full_oof)

blend_full_oof = 0.5*(oof_full_matrix @ best_full_w) + 0.5*ridge_meta_full.predict(oof_full_matrix)
blend_full_oof_mse = mean_squared_error(y_full, blend_full_oof)

print(f'\nPhase 4 OOF MSE (SLSQP): {final_full_oof_mse:.3f}')
print(f'Phase 4 OOF MSE (Blend): {blend_full_oof_mse:.3f}')

if blend_full_oof_mse < final_full_oof_mse:
    final_test_phase4 = 0.5*(test_full_matrix @ best_full_w) + 0.5*ridge_meta_full.predict(test_full_matrix)
    final_phase4_label = 'Blend'
else:
    final_test_phase4 = test_full_matrix @ best_full_w
    final_phase4_label = 'SLSQP'
final_test_phase4 = np.clip(final_test_phase4, 0, 100)

## 13. 🏁 Final Submission

In [ ]:
# Phase 1 (dev only) vs Phase 4 (full) blend (en güvenli)
final_test = 0.5 * final_test_phase1 + 0.5 * final_test_phase4
final_test = np.clip(final_test, 0, 100)

submission = pd.DataFrame({
    'student_id'         : test_ids,
    'career_success_score': final_test
})
submission.to_csv('submission.csv', index=False)

# Bonus: alternatif submissions kaydet
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase1}).to_csv('submission_phase1.csv', index=False)
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase4}).to_csv('submission_phase4.csv', index=False)

print('═'*60)
print('  🏆 v5 FINAL RESULTS')
print('═'*60)
print(f'  Phase 1 (dev only) Holdout MSE : {best_holdout_mse:.3f}')
print(f'  Phase 4 (full)     OOF MSE     : {min(final_full_oof_mse, blend_full_oof_mse):.3f}')
print(f'  Final blend                    : 50% Phase1 + 50% Phase4')
print('-'*60)
print(f'  Leaderboard referans:')
print(f'    1. sıra (Berkay Fehmi Tekin) : 83.42')
print(f'    10. sıra (CodeCosmos)        : 84.79')
print(f'    Sen (v1)                     : 89.54')
print(f'    v5 beklenen                  : {best_holdout_mse:.2f}-{min(final_full_oof_mse, blend_full_oof_mse):.2f}')
print('═'*60)
print(f'\n3 dosya üretildi:')
print(f'  - submission.csv         (Phase1 + Phase4 blend - ÖNCE BUNU SUBMIT ET)')
print(f'  - submission_phase1.csv  (dev only, validated on holdout)')
print(f'  - submission_phase4.csv  (full data retrained)')
print(f'\n{submission.head()}')

In [ ]:
# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Per-model dev OOF MSE vs Holdout MSE
dev_mses = [mean_squared_error(y_dev, oof_matrix[:,i]) for i in range(n_models)]
hold_mses = [mean_squared_error(y_hold, holdout_matrix[:,i]) for i in range(n_models)]
x = np.arange(len(model_names))
axes[0,0].bar(x - 0.2, dev_mses, 0.4, label='Dev OOF MSE', color='steelblue')
axes[0,0].bar(x + 0.2, hold_mses, 0.4, label='Holdout MSE', color='coral')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(model_names)
axes[0,0].legend(); axes[0,0].set_title('OOF vs Holdout — Per Model')

# Dağılım
axes[0,1].hist(y_full, bins=50, alpha=0.6, color='steelblue', label='Train', edgecolor='white')
axes[0,1].hist(final_test, bins=50, alpha=0.6, color='coral', label='Test (final)', edgecolor='white')
axes[0,1].set_title('Dağılım'); axes[0,1].legend()

# Stability
fold_data = []
for name, fmse in stability.items():
    for v in fmse: fold_data.append({'Model': name, 'MSE': v})
fold_df = pd.DataFrame(fold_data)
sns.boxplot(data=fold_df, x='Model', y='MSE', ax=axes[1,0])
axes[1,0].set_title('Per-Fold MSE Stability')

# Weight pie
axes[1,1].pie(best_full_w, labels=model_names, autopct='%1.1f%%')
axes[1,1].set_title('Phase 4 Ensemble Weights')

plt.tight_layout(); plt.show()

---
## 📋 v5 Mimari Özet

| Bölüm | İçerik |
|-------|--------|
| **Validation** | 80/20 stratified hold-out + 10-fold CV |
| **Feature Eng.** | Role-skill + numerical aggregates + cross TE |
| **NLP** | TF-IDF (word+char) + Sentence-BERT (multilingual) |
| **Feature Selection** | XGB importance — top %85 tree, top 30 KNN, top 10 Ridge |
| **Base Models** | 7 model, 4 algoritma türü |
| **Hiperparametre** | Optuna 60/60/40/25/25/15/15 trial |
| **Stacking** | 10-fold × 3 seed, OOF + Holdout + Test |
| **Meta-learning** | SLSQP + Ridge + Blend (holdout MSE en iyisi seçilir) |
| **Pseudo-labeling** | %10 konservatif, A/B test on holdout |
| **Final** | Phase1 (dev) + Phase4 (full) blend |

## 🚀 Çalıştırma Süresi (GPU ile tahmin)
- BERT + features: 10 dk
- Optuna (260 trial): ~60 dk
- Phase 1 stacking: ~30 dk
- Phase 3 pseudo stacking: ~30 dk
- Phase 4 full stacking: ~30 dk
- **Toplam: ~2.5-3 saat**

## 🎯 Beklenen Sonuç
- v1 → 89.54 MSE (117. sıra)
- **v5 → ~75-80 MSE (ilk 5-10)**